# Finite weighted-decay block access

**Status: source prepared; execution pending.** This companion concerns one original primitive block oracle. Its selected block is the finite realized weighted decay \(D_{\rm fin}/\alpha\); it does not implement the coherent correction, its evolution, a channel or Gibbs preparation.

The caller supplies the system input and zero clock, proposal, weight and selector registers. The selected output requires **every auxiliary register to be zero**. Both complete Gaussian access streams, their inverses and the weight rotations belong to the logical input-access cost. Standalone initialization, selection repetitions/amplification, measurement, synthesis and physical faults need separate accounting.

The later authorized notebook execution draws a source circuit and reads the previously exported three-case records. It does not rerun their statevector/reference checks. The underlying Gaussian-weighted decay and classical spectral correction are connected to Chen and colleagues, *Efficient quantum thermal simulation*, Nature 646 (2025), DOI 10.1038/s41586-025-09583-x, equations 7, 8 and 19. Construction is original, using the project MIT sources and exact locked Qiskit/NumPy/SciPy environment.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys

sys.dont_write_bytecode = True
source_root = os.environ.get("OPENQUANTUM_SOURCE_ROOT")
root = Path(source_root).resolve() if source_root else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/openquantum").is_dir()
)
sys.path.insert(0, str(root / "src"))
from openquantum.simulation import detailed_balance_decay as decay
assert Path(decay.__file__).resolve() == (
    root / "src/openquantum/simulation/detailed_balance_decay.py"
).resolve()

MAX_BUNDLE_BYTES = 64 * 1024 * 1024
source_inputs = {
    "detailed_balance_decay.py": root / "src/openquantum/simulation/detailed_balance_decay.py",
    "detailed_balance_decay_reference.py": root / "src/openquantum/simulation/detailed_balance_decay_reference.py",
    "detailed_balance_access.py": root / "src/openquantum/simulation/detailed_balance_access.py",
    "generate_detailed_balance_decay.py": root / "scripts/generate_detailed_balance_decay.py",
    "requirements-lock.txt": root / "requirements-lock.txt",
    "LICENSE.txt": root / "LICENSE",
}

def read_saved_bundle(path_variable, hash_variable, expected_status):
    report_path = Path(os.environ[path_variable]).resolve()
    expected_hash = os.environ[hash_variable]
    if not re.fullmatch(r"[0-9a-f]{64}", expected_hash):
        raise ValueError("a trusted manifest SHA256 is required")
    if report_path.name != "results.json":
        raise ValueError("the saved report must be results.json")
    directory = report_path.parent
    manifest_path = directory / "manifest.json"
    sidecar_path = directory / "manifest.sha256"
    for path in (manifest_path, sidecar_path):
        if not path.is_file() or path.is_symlink() or path.stat().st_size > MAX_BUNDLE_BYTES:
            raise ValueError("manifest inputs must be bounded regular files")
    manifest_raw = manifest_path.read_bytes()
    if hashlib.sha256(manifest_raw).hexdigest() != expected_hash:
        raise ValueError("the manifest differs from the supplied identity pin")
    if sidecar_path.read_bytes() != (expected_hash + "  manifest.json\n").encode("ascii"):
        raise ValueError("the manifest sidecar differs from its pin")
    manifest = json.loads(manifest_raw)
    files = manifest["files"]
    if not isinstance(files, dict) or not 1 <= len(files) <= 32:
        raise ValueError("the saved manifest inventory is not bounded")
    if "manifest.json" in files or "manifest.sha256" in files or "results.json" not in files:
        raise ValueError("manifest and sidecar must be separate from payloads")
    actual_names = set()
    total_bytes = len(manifest_raw) + sidecar_path.stat().st_size
    for path in directory.iterdir():
        if not path.is_file() or path.is_symlink():
            raise ValueError("the saved bundle must contain only regular flat files")
        actual_names.add(path.name)
    if actual_names != set(files) | {"manifest.json", "manifest.sha256"}:
        raise ValueError("the saved bundle has missing or extra files")
    for name, pin in files.items():
        if not isinstance(name, str) or Path(name).name != name or name in ("", ".", ".."):
            raise ValueError("manifest payload names must be flat")
        if not isinstance(pin, dict) or set(pin) != {"bytes", "sha256"}:
            raise ValueError("payload pins must contain bytes and SHA256")
        if type(pin["bytes"]) is not int or not 0 <= pin["bytes"] <= MAX_BUNDLE_BYTES:
            raise ValueError("payload sizes must be bounded integers")
        if not isinstance(pin["sha256"], str) or not re.fullmatch(r"[0-9a-f]{64}", pin["sha256"]):
            raise ValueError("payload SHA256 pins must be typed")
        total_bytes += pin["bytes"]
        if total_bytes > MAX_BUNDLE_BYTES:
            raise ValueError("the saved bundle exceeds its complete output cap")
        payload_path = directory / name
        if payload_path.stat().st_size != pin["bytes"]:
            raise ValueError("saved payload size differs before its bounded read")
        raw = payload_path.read_bytes()
        if len(raw) != pin["bytes"] or hashlib.sha256(raw).hexdigest() != pin["sha256"]:
            raise ValueError("saved payload bytes differ from the manifest")
    if manifest["total_payload_bytes"] != sum(pin["bytes"] for pin in files.values()):
        raise ValueError("the saved payload byte total differs")
    for name, source in source_inputs.items():
        if not source.is_file() or source.is_symlink() or source.stat().st_size != files[name]["bytes"]:
            raise ValueError("current source size differs before its bounded read")
        raw = source.read_bytes()
        if len(raw) != files[name]["bytes"] or hashlib.sha256(raw).hexdigest() != files[name]["sha256"]:
            raise ValueError("saved source bytes differ from this notebook's selected worktree")
    report = json.loads(report_path.read_bytes())
    if report["status"] != expected_status or manifest["status"] != expected_status:
        raise ValueError("the saved report has the wrong execution scope")
    for flag in ("all_components_verified", "new_algorithm_result", "coherent_correction_implemented",
                 "coherent_correction_evolution_implemented", "general_channel_implemented",
                 "sampler_implemented", "rotations_synthesized", "physical_resources_computed"):
        if report[flag] is not False or manifest[flag] is not False:
            raise ValueError("the saved report exceeds the decay-block scope")
    return report


## Primitive source illustration

The selector Hadamards combine the identity branch with \(-V^\dagger Z_wV\), where \(V=R_wU\). The selector phase \(P(\pi)\) supplies the minus sign. With all auxiliaries zero, the resulting block is \(D_{\rm fin}/\alpha\), where \(\alpha=\eta\max_l\gamma_l\). Selecting only the selector wire would leave additional environment outputs.

The following source illustration uses the frozen diagonal input \(H=Z\), proposals \(X,Z\), \(\beta=1\), two clock bits and one product step. Building and drawing it performs no statevector or density-matrix calculation.

In [2]:
illustration = decay.build_decay_block(
    ((1.0, "Z"),), ("X", "Z"), beta=1.0, clock_bits=2, product_steps=1
)
assert illustration.num_qubits == 6
print(illustration.draw("text", fold=100))
print(json.dumps({
    "source_operations": illustration.metadata["source_operations"],
    "access_operations_each_copy": illustration.metadata["access_source_operations_each_copy"],
    "selected_auxiliary_state": illustration.metadata["selected_auxiliary_state"],
    "numerical_checks_performed": illustration.metadata["numerical_checks_performed"],
}, indent=2))


     ┌─────────────┐                          ┌───┐┌──────────────┐┌───┐┌─────────────┐┌───┐»
q_0: ┤ Rz(0.31333) ├──────────────────────────┤ X ├┤ Rz(-0.31333) ├┤ X ├┤ Rz(0.31333) ├┤ X ├»
     └┬───────────┬┘┌───┐┌──────────────┐┌───┐└─┬─┘└──────────────┘└─┬─┘└─────────────┘└─┬─┘»
q_1: ─┤ Ry(1.866) ├─┤ X ├┤ Ry(-0.67823) ├┤ X ├──■────────────────────■───────────────────■──»
      ├───────────┴┐└─┬─┘└──────────────┘└─┬─┘                                              »
q_2: ─┤ Ry(1.0594) ├──■────────────────────■────────────────────────────────────────────────»
      └───┬───┬────┘┌───┐                                                                   »
q_3: ─────┤ H ├─────┤ X ├───────────────────────────────────────────────────────────────────»
      ┌───┴───┴────┐└───┘                                                                   »
q_4: ─┤ Ry(2.1293) ├────────────────────────────────────────────────────────────────────────»
      └───┬───┬────┘                                        

## Read the saved three-case evidence

The later execution requires explicit paths and trusted manifest pins for the exported grid. Each payload, copied source, lock and license is authenticated before display. These cells read retained records; they do not perform new numerical recovery or replace the finite product-formula target with exact Hamiltonian evolution.

In [3]:
grid = read_saved_bundle(
    "OPENQUANTUM_DECAY_RESULTS", "OPENQUANTUM_DECAY_MANIFEST_SHA256",
    "checked_small_finite_decay_block"
)
assert [record["name"] for record in grid["cases"]] == ["diagonal", "tilted", "complex"]
assert [record["metadata"]["total_wires"] for record in grid["cases"]] == [6, 5, 8]
for record in grid["cases"]:
    assert record["numerical_checks_performed"] is True
    assert (record["beta"], record["clock_bits"], record["product_steps"]) == (1.0, 2, 1)
    assert record["selected_auxiliary_state"] == "all_zero"
print(json.dumps({
    "status": grid["status"],
    "source_commit": grid["source_commit"],
    "source_revision_is_exact": grid["source_revision_is_exact"],
    "actual_widths": [record["metadata"]["total_wires"] for record in grid["cases"]],
    "scope": grid["scope"],
}, indent=2))


{
  "status": "checked_small_finite_decay_block",
  "source_commit": "b1a05ef76a47d8e7ef893cd6bf0ffb66772d4ce1",
  "source_revision_is_exact": true,
  "actual_widths": [
    6,
    5,
    8
  ],
  "scope": "finite-Trotter weighted-decay block access; every selected auxiliary wire is zero"
}


In [4]:
for record in grid["cases"]:
    check = record["check"]
    print(record["name"])
    print(json.dumps({
        "actual_vs_finite_trotter_max_entry_error": check["actual_vs_finite_trotter_max_entry_error"],
        "actual_vs_finite_trotter_frobenius_error": check["actual_vs_finite_trotter_frobenius_error"],
        "actual_vs_exact_finite_frobenius_error": check["actual_vs_exact_finite_frobenius_error"],
        "finite_trotter_vs_exact_finite_frobenius_error": check["finite_trotter_vs_exact_finite_frobenius_error"],
        "normalization": check["normalization"],
        "finite_diagnostics": check["finite_diagnostics"],
        "actual_diagnostics": check["actual_diagnostics"],
        "actual_selected_block": check["matrices"]["actual_selected_block"],
        "independent_finite_trotter_normalized_decay": check["matrices"]["finite_trotter_normalized_decay"],
    }, indent=2, allow_nan=False))


diagonal
{
  "actual_vs_finite_trotter_max_entry_error": 1.1103068243156721e-15,
  "actual_vs_finite_trotter_frobenius_error": 1.176590116471956e-15,
  "actual_vs_exact_finite_frobenius_error": 1.4231351965542707e-15,
  "finite_trotter_vs_exact_finite_frobenius_error": 2.482534153247273e-16,
  "normalization": {
    "independent_eta": 0.9775450868978823,
    "independent_gamma_max": 1.0,
    "independent_alpha": 0.9775450868978823,
    "builder_eta": 0.9775450868978823,
    "builder_gamma_max": 1.0,
    "builder_alpha": 0.9775450868978823,
    "eta_absolute_error": 0.0,
    "gamma_max_absolute_error": 0.0,
    "alpha_absolute_error": 0.0,
    "builder_alpha_product_residual": 0.0,
    "dt_dw_residual": 2.220446049250313e-16,
    "physical_squared_sum": 1.5599362646490316,
    "normalized_gamma": [
      0.6065306597126334,
      0.04945815726640235,
      1.0,
      1.0
    ],
    "scope": "eta=dt*sum(f_j**2) independently; eta does not disappear from physical alpha"
  },
  "finite_dia

## Logical access cost and scalable source

The independent physical normalization check retains the mesh mass \(\eta\); it cannot be inferred merely by comparing normalized matrices. The raw record keeps complex entries, Hermiticity and eigenvalue diagnostics without clipping tiny negative roundoff. Exact-H finite-target differences have their own fields and are not an acceptance target for the realized product-formula oracle.

The next cell reads the separate twelve-spin, eighteen-wire source-only export. Its count includes both actual access streams, both weight multiplexors, the selector reflection and full inverses. It performs no large-system simulation, input preparation, readout or amplification. A future complete algorithm must add those costs and rotation synthesis, physical faults and classical processing. No surface-code comparison or mixing claim follows from this source.

In [5]:
large_source = read_saved_bundle(
    "OPENQUANTUM_DECAY_SOURCE_RESULTS", "OPENQUANTUM_DECAY_SOURCE_MANIFEST_SHA256",
    "generated_decay_block_without_simulation"
)
assert len(large_source["cases"]) == 1
large = large_source["cases"][0]
assert large["name"] == "chain-n12"
assert large["metadata"]["n_system"] == 12
assert large["metadata"]["total_wires"] == 18
assert large["numerical_checks_performed"] is False
assert "check" not in large
print(json.dumps({
    "name": large["name"],
    "source_operations": large["metadata"]["source_operations"],
    "primitive_counts": large["primitive_counts"],
    "resource_ledger": large["metadata"]["resource_ledger"],
    "all_components_verified": large_source["all_components_verified"],
    "new_algorithm_result": large_source["new_algorithm_result"],
    "remaining": large_source["remaining"],
}, indent=2, allow_nan=False))


{
  "name": "chain-n12",
  "source_operations": 2408,
  "primitive_counts": {
    "cx": 1159,
    "rz": 736,
    "h": 420,
    "tdg": 28,
    "t": 28,
    "x": 16,
    "ry": 14,
    "p": 7
  },
  "resource_ledger": {
    "access_copies": 2,
    "access_instructions_each_copy": 1193,
    "access_instructions_total": 2386,
    "proposal_prepare_h_instructions_total": 4,
    "gaussian_prepare_instructions_total": 10,
    "pauli_select_instructions_total": 144,
    "signed_evolution_instructions_total": 2208,
    "dft_instructions_total": 20,
    "requested_signed_interval_magnitude_all_access_directions": 7.519884823893001,
    "weight_multiplexor_copies": 2,
    "weight_ry_instructions_total": 8,
    "weight_cx_instructions_total": 8,
    "selector_and_reflection_instructions_total": 6,
    "source_operations_total": 2408,
    "classical_preprocessing": "explicit Gaussian/prefix masses and naive Gray-Walsh angle transform: O(M^2) time, O(M) storage; wall time is not quantum runtime",
   

The saved small records address finite weighted-decay block access and its independent reference. The noncommuting coherent correction and its evolution, full channel, continuum/KMS accuracy, mixing/output guarantees, synthesis, complete initialization/readout/repetition costs, physical resources and mechanized proof remain planned. Evidence and accounting belong inline in the detailed-balance Gibbs article; this companion adds no separate cost or verification chapter.